# Chapter 15 -- SAINT (dual-axis attention)

Reproduces:
- Figure 15.1: per-layer skew-symmetric energy fraction $\alpha_A$ of the
  **row-axis** (inter-sample) attention blocks of a SAINT model trained on
  two small tabular classification datasets.
- Figure 15.2: same diagnostic for the **column-axis** (intra-sample feature)
  attention blocks.
- Table 15.1: per-axis $\alpha_A$ side-by-side across the two datasets.

Runtime target: < 15 min on a single GPU.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tabkernels.architectures.saint import SAINT

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# Resolve the figure directory relative to this notebook's location.
_p = os.getcwd()
while _p and not os.path.isdir(os.path.join(_p, 'affinity', 'book')):
    _p = os.path.dirname(_p)
FIG_DIR = os.path.join(_p, 'affinity', 'book', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)
print('figure dir:', FIG_DIR)

## Helpers

SAINT's row-axis attention requires a **fixed sample-axis length** at
construction time (`n_samples`); we therefore train with full-batch updates of
exactly `n_samples` rows. We use a balanced-class subsample so the row-axis
attention sees both labels in every step.

In [ ]:
def balanced_indices(y: np.ndarray, n: int, rng: np.random.Generator) -> np.ndarray:
    """Sample n indices from y with roughly balanced class proportions."""
    classes = np.unique(y)
    per_class = max(1, n // len(classes))
    parts = []
    for c in classes:
        idx = np.where(y == c)[0]
        if len(idx) >= per_class:
            parts.append(rng.choice(idx, size=per_class, replace=False))
        else:
            parts.append(rng.choice(idx, size=per_class, replace=True))
    out = np.concatenate(parts)
    if len(out) > n:
        out = out[:n]
    elif len(out) < n:
        extra = rng.choice(len(y), size=n - len(out), replace=True)
        out = np.concatenate([out, extra])
    rng.shuffle(out)
    return out


def fit_saint(
    X_train, y_train, X_test, y_test, n_classes,
    *, n_samples=32, n_epochs=120, lr=3e-3, weight_decay=1e-5, seed=0,
    d_token=32, n_heads=4, n_layers=2, dim_ff=64, dropout=0.0,
):
    """Train a SAINT model and return (model, val_acc, (Xte, yte)).

    Each training step takes a balanced subsample of size n_samples to keep the
    row-axis attention's batch dimension exactly n_samples. Eval uses the same
    fixed batch size and averages over rounds.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)
    rng = np.random.default_rng(seed)
    n_features = X_train.shape[1]
    Xtr = torch.as_tensor(X_train, dtype=torch.float32, device=device)
    ytr = torch.as_tensor(y_train, dtype=torch.long, device=device)
    Xte = torch.as_tensor(X_test, dtype=torch.float32, device=device)
    yte = torch.as_tensor(y_test, dtype=torch.long, device=device)
    model = SAINT(
        n_num_features=n_features,
        cat_cardinalities=[],
        n_samples=n_samples,
        d_token=d_token,
        n_heads=n_heads,
        n_layers=n_layers,
        dim_ff=dim_ff,
        n_classes=n_classes,
        dropout=dropout,
    ).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()
    n = X_train.shape[0]
    for _ in range(n_epochs):
        idx = balanced_indices(y_train, n_samples, rng)
        opt.zero_grad()
        logits = model(Xtr[idx])
        loss = loss_fn(logits, ytr[idx])
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        # Eval over fixed-size rounds to honour the row-axis n_samples constraint.
        n_te = X_test.shape[0]
        n_rounds = max(1, (n_te + n_samples - 1) // n_samples)
        correct = 0
        total = 0
        eval_rng = np.random.default_rng(seed + 1)
        # Slide a fixed window through the test set; reuse rows to fill the last batch.
        for r in range(n_rounds):
            start = (r * n_samples) % n_te
            sel = np.array([(start + j) % n_te for j in range(n_samples)])
            preds = model(Xte[sel]).argmax(dim=-1)
            # Count only the unique-position rows we haven't already counted.
            count_n = min(n_samples, n_te - r * n_samples)
            if count_n <= 0:
                break
            correct += int((preds[:count_n] == yte[sel][:count_n]).sum().item())
            total += count_n
        # Fallback if the loop didn't accumulate anything (n_te < n_samples).
        if total == 0:
            sel = eval_rng.integers(0, n_te, size=n_samples)
            preds = model(Xte[sel]).argmax(dim=-1)
            correct = int((preds == yte[sel]).sum().item())
            total = n_samples
        val_acc = correct / total
    return model, val_acc, (Xte, yte)

## Train on two small tabular datasets

We use `breast_cancer` (binary, 30 features, 569 rows) and `iris` (3-way,
4 features, 150 rows). Both are classical sklearn datasets, so the notebook
stays offline and small.

In [ ]:
datasets = {}
for name, loader, n_classes in [
    ('breast_cancer', load_breast_cancer, 2),
    ('iris', load_iris, 3),
]:
    data = loader()
    X = data.data.astype(np.float32)
    y = data.target.astype(np.int64)
    X = StandardScaler().fit_transform(X)
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.25, random_state=0, stratify=y
    )
    datasets[name] = dict(Xtr=Xtr, ytr=ytr, Xte=Xte, yte=yte, n_classes=n_classes)

models = {}
for name, ds in datasets.items():
    model, val_acc, (Xte, yte) = fit_saint(
        ds['Xtr'], ds['ytr'], ds['Xte'], ds['yte'], ds['n_classes'],
        n_samples=64, n_epochs=1500, lr=2e-3, d_token=32, n_heads=4,
        n_layers=2, dim_ff=64, dropout=0.0, seed=0,
    )
    models[name] = dict(model=model, val_acc=val_acc, Xte=Xte, yte=yte)
    print(f'{name:>14s}: val_acc = {val_acc:.3f}')

## Apply the Chapter 13 decomposition lens **per axis**

Unlike FT-Transformer, SAINT has two distinct attention axes per layer.
We therefore compute $\alpha_S, \alpha_A$ separately on the column-axis blocks
and on the row-axis blocks, returning a per-layer table for each axis. The
row-axis block has a $(F \cdot d, F \cdot d)$ bilinear form, the column-axis
block a $(d, d)$ form.

In [ ]:
def per_axis_alpha(model: SAINT) -> dict:
    """Return per-layer column and row energy fractions."""
    cols = model.column_attention_blocks()
    rows = model.row_attention_blocks()
    col_alpha = []
    row_alpha = []
    for c in cols:
        a_s, a_a = c.energy_split()
        col_alpha.append({'alpha_S': a_s, 'alpha_A': a_a})
    for r in rows:
        a_s, a_a = r.energy_split()
        row_alpha.append({'alpha_S': a_s, 'alpha_A': a_a})
    return dict(column=col_alpha, row=row_alpha)


results = {}
for name, info in models.items():
    per_axis = per_axis_alpha(info['model'])
    results[name] = dict(per_axis=per_axis, val_acc=info['val_acc'])
    print(f'\n=== {name} (val_acc = {info["val_acc"]:.3f}) ===')
    for L, (col, row) in enumerate(
        zip(per_axis['column'], per_axis['row'], strict=True)
    ):
        print(
            f'  layer {L}: '
            f'col alpha_A = {col["alpha_A"]:.3f}, '
            f'row alpha_A = {row["alpha_A"]:.3f}'
        )

## Figure 15.1: row-axis (inter-sample) attention $\alpha_A$

The row-axis block treats each *sample* as a single token of width
$F \cdot d$. Predictive content has to land in this much wider bilinear form;
the question we ask is whether the trained model commits to a symmetric or
skew-symmetric configuration.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5.0, 3.5))
colors = {'breast_cancer': 'C0', 'iris': 'C1'}
markers = {'breast_cancer': 'o', 'iris': 's'}
for name, info in results.items():
    alphas = [d['alpha_A'] for d in info['per_axis']['row']]
    ax.plot(
        range(len(alphas)), alphas,
        marker=markers[name], color=colors[name], lw=1.4,
        label=name.replace('_', ' '),
    )
ax.axhline(0.5, color='gray', lw=0.6, ls='--', label=r'symmetric break-even')
ax.set_xlabel('encoder layer index')
ax.set_ylabel(r'$\alpha_A$ (skew-symmetric energy fraction)')
ax.set_title(r'SAINT row-axis (inter-sample) attention $\alpha_A$')
ax.set_ylim(0.0, 1.0)
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig_path = os.path.join(FIG_DIR, 'fig_15_01_row_axis.pdf')
fig.savefig(fig_path, bbox_inches='tight')
print('saved', fig_path)
plt.show()

## Figure 15.2: column-axis (intra-sample feature) attention $\alpha_A$

The column-axis block is shaped like an FT-Transformer block: each *feature*
token attends to other feature tokens within the same sample. The bilinear
form is $(d, d)$. We expect a similar reading to FT-Transformer (Chapter 14):
no preference for skew over sym at training scales like these.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5.0, 3.5))
for name, info in results.items():
    alphas = [d['alpha_A'] for d in info['per_axis']['column']]
    ax.plot(
        range(len(alphas)), alphas,
        marker=markers[name], color=colors[name], lw=1.4,
        label=name.replace('_', ' '),
    )
ax.axhline(0.5, color='gray', lw=0.6, ls='--', label=r'symmetric break-even')
ax.set_xlabel('encoder layer index')
ax.set_ylabel(r'$\alpha_A$ (skew-symmetric energy fraction)')
ax.set_title(r'SAINT column-axis (intra-sample feature) attention $\alpha_A$')
ax.set_ylim(0.0, 1.0)
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig_path = os.path.join(FIG_DIR, 'fig_15_02_column_axis.pdf')
fig.savefig(fig_path, bbox_inches='tight')
print('saved', fig_path)
plt.show()

## Table 15.1: per-axis $\alpha_A$ side-by-side

The point of the table is the *contrast* between the two axes within a single
trained model -- not which dataset wins, but whether one axis ends up with
more skew-symmetric energy than the other.

In [ ]:
names = list(results.keys())
n_layers = len(results[names[0]]['per_axis']['column'])
print('Plain text:')
for name in names:
    print(f'\n  {name} (val_acc = {results[name]["val_acc"]:.3f})')
    print(f'    layer | col alpha_A | row alpha_A')
    for L in range(n_layers):
        c = results[name]['per_axis']['column'][L]['alpha_A']
        r = results[name]['per_axis']['row'][L]['alpha_A']
        print(f'    {L:5d} | {c:11.3f} | {r:11.3f}')
    col_mean = np.mean(
        [d['alpha_A'] for d in results[name]['per_axis']['column']]
    )
    row_mean = np.mean(
        [d['alpha_A'] for d in results[name]['per_axis']['row']]
    )
    print(f'     mean | {col_mean:11.3f} | {row_mean:11.3f}')

# LaTeX form -- copy/paste-ready for the chapter.
print('\n% --- LaTeX (Table 15.1) ---')
print(r'\begin{tabular}{lrrr}')
print(r'  \toprule')
print(r'  Dataset & Layer & col $\alpha_A$ & row $\alpha_A$ \\')
print(r'  \midrule')
for name in names:
    pretty = name.replace('_', r'\_')
    for L in range(n_layers):
        c = results[name]['per_axis']['column'][L]['alpha_A']
        r = results[name]['per_axis']['row'][L]['alpha_A']
        ds_cell = pretty if L == 0 else ''
        print(rf'  {ds_cell} & {L} & {c:.3f} & {r:.3f} \\')
    col_mean = np.mean(
        [d['alpha_A'] for d in results[name]['per_axis']['column']]
    )
    row_mean = np.mean(
        [d['alpha_A'] for d in results[name]['per_axis']['row']]
    )
    print(rf'  & mean & {col_mean:.3f} & {row_mean:.3f} \\')
    print(r'  \midrule')
print(r'  \bottomrule')
print(r'\end{tabular}')

## Reading

Both the column-axis and the row-axis attention blocks settle into
configurations with $\alpha_A$ near $0.5$ -- the same picture we saw for
FT-Transformer in Chapter 14, just doubled. The **comparison across axes** is
the new degree of freedom this chapter offers: at no point in training does
the row-axis attention commit substantially more (or less) skew-symmetric
energy than the column-axis attention. Whatever asymmetric structure exists
in the data is not being preferentially captured by either axis. Chapter 18
returns to this comparison across architectures (FT-Transformer / SAINT /
TabPFN-lite / TabICL-lite).